#Install Dependencies

In [139]:
# Install all required libraries
!apt-get install -y poppler-utils
!pip install -qU langchain-groq langgraph langchain-community pysqlite3-binary pdfplumber langchain-huggingface chromadb sentence-transformers pdf2image langchain-text-splitters pandas

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


# Environment Setup

In [140]:
import os
from google.colab import userdata

# Set your Groq API Key in Colab Secrets (the key icon on the left)
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Environment ready.")

Environment ready.


#Document Loading & File Upload

In [141]:
from google.colab import files
import glob

# 1. Trigger the upload widget
uploaded = files.upload()

# 2. Store only the names of files uploaded in this specific session
DEMO_FILES = list(uploaded.keys())

if not DEMO_FILES:
    print("No files uploaded. Please rerun this cell.")
else:
    print(f"Successfully loaded {len(DEMO_FILES)} files for processing.")

Saving 04_defamation_online_review.pdf to 04_defamation_online_review (3).pdf
Saving 05_patent_infringement_medical_device.pdf to 05_patent_infringement_medical_device (4).pdf
Saving bw_doc_3.pdf to bw_doc_3 (5).pdf
Saving bw_doc_4.pdf to bw_doc_4 (8).pdf
Saving LOA5.pdf to LOA5 (3).pdf
Saving LOA7.pdf to LOA7 (1).pdf
Saving notice_1.pdf to notice_1 (1).pdf
Saving notice_4.pdf to notice_4 (2).pdf
Successfully loaded 8 files for processing.


#RAG & Audit Setup

In [142]:
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Setup RAG Knowledge Base (The "Brain")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
knowledge = [
    "A Cease and Desist letter must demand the recipient stop illegal activity like infringement.",
    "A Letter of Authority (LoA) or Limited Power of Attorney is NOT a Cease and Desist request.",
    "General notices or debt resolution updates are Irrelevant to Cease and Desist processing."
]
docs = RecursiveCharacterTextSplitter(chunk_size=500).create_documents(knowledge)
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# 2. Setup Audit Agent
class AuditAgent:
    def log(self, file_name, action, result):
        entry = pd.DataFrame([{
            'Timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'File': file_name, 'Action': action, 'Result': result
        }])
        entry.to_csv("audit_log.csv", mode='a', header=not os.path.exists("audit_log.csv"), index=False)

audit_agent = AuditAgent()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Agent Definitions (Logic & Agents)

In [143]:
import sqlite3
import csv
import datetime
import pdfplumber
import pandas as pd
from io import BytesIO
from pdf2image import convert_from_path
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 1. Audit & Database Agent ---
class AuditAgent:
    def __init__(self, log_file="audit_log.csv"):
        self.log_file = log_file
        # Create CSV with headers if it doesn't exist
        if not os.path.exists(self.log_file):
            pd.DataFrame(columns=['Timestamp', 'File', 'Action', 'Result']).to_csv(self.log_file, index=False)

    def log(self, file_name, action, result):
        entry = pd.DataFrame([{
            'Timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'File': file_name, 'Action': action, 'Result': result
        }])
        entry.to_csv(self.log_file, mode='a', header=False, index=False)

# --- 2. Document Loader Agent ---
class DocumentLoaderAgent:
    def load(self, file_path):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
        return text

# --- 3. RAG Classification Agent (Fixed) ---
class RAGClassificationAgent:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm

    def classify(self, doc_text):
        # Fetch legal context from RAG
        relevant_docs = self.retriever.invoke(doc_text)
        context = "\n".join([d.page_content for d in relevant_docs])

        prompt = f"""
        LEGAL CONTEXT: {context}
        DOCUMENT TO ANALYZE: {doc_text[:1500]}

        INSTRUCTION: Based on the context, is this a 'Cease' and Desist, an 'Irrelevant' document (like an LoA), or are you 'Uncertain'?
        Respond with ONLY one word: Cease, Irrelevant, or Uncertain.
        """
        response = self.llm.invoke(prompt).content.strip().capitalize()
        return response if response in ["Cease", "Irrelevant", "Uncertain"] else "Uncertain"

rag_classifier = RAGClassificationAgent(retriever, llm)

#Initialization

In [144]:
# Initialize LLM
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0)

# Initialize RAG Knowledge Base
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
knowledge = ["Cease and Desist requests must stop illegal activity.",
             "Letters of Authority (LoA) are standard legal notices, not Cease/Desist."]
docs = RecursiveCharacterTextSplitter(chunk_size=500).create_documents(knowledge)
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# Instantiate Agents
loader = DocumentLoaderAgent()
rag_classifier = RAGClassificationAgent(retriever, llm)
audit_agent = AuditAgent()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Main Execution Loop

In [145]:
print("Starting RAG-Augmented Processing...")

for file_path in DEMO_FILES:
    try:
        raw_text = extract_text_from_pdf(file_path)
        result = rag_classifier.classify(raw_text)

        print(f"\n>>> File: {file_path} | AI Result: {result}")

        if result == "Uncertain":
            # This will now trigger because the prompt is stricter
            final_decision = input(f"MANUAL REVIEW for {file_path}: Enter 'Cease' or 'Irrelevant': ").capitalize()
            audit_agent.log(file_path, "Manual Review", final_decision)
        else:
            audit_agent.log(file_path, "Auto-Classification", result)

    except Exception as e:
        print(f"Error: {e}")

Starting RAG-Augmented Processing...

>>> File: 04_defamation_online_review (3).pdf | AI Result: Uncertain
MANUAL REVIEW for 04_defamation_online_review (3).pdf: Enter 'Cease' or 'Irrelevant': Cease

>>> File: 05_patent_infringement_medical_device (4).pdf | AI Result: Uncertain
MANUAL REVIEW for 05_patent_infringement_medical_device (4).pdf: Enter 'Cease' or 'Irrelevant': Irrelevant

>>> File: bw_doc_3 (5).pdf | AI Result: Cease

>>> File: bw_doc_4 (8).pdf | AI Result: Cease

>>> File: LOA5 (3).pdf | AI Result: Uncertain
MANUAL REVIEW for LOA5 (3).pdf: Enter 'Cease' or 'Irrelevant': Cease

>>> File: LOA7 (1).pdf | AI Result: Uncertain
MANUAL REVIEW for LOA7 (1).pdf: Enter 'Cease' or 'Irrelevant': Cease

>>> File: notice_1 (1).pdf | AI Result: Cease

>>> File: notice_4 (2).pdf | AI Result: Cease


#Display Audit Logs

In [146]:
# View the audit trail
history = pd.read_csv("audit_log.csv")
display(history)

,timestamp,document_name,action,explanation
0,2026-03-20T13:50:21.294055,10_breach_of_contract_nda.pdf,Loaded,Document loaded and text extracted.
1,2026-03-20T13:50:43.315245,10_breach_of_contract_nda.pdf,Loaded,Document loaded and text extracted.
2,2026-03-20T13:50:58.805887,10_breach_of_contract_nda.pdf,Loaded,Document loaded and text extracted.
3,2026-03-20T13:53:38.788413,10_breach_of_contract_nda.pdf,Loaded,Document loaded and text extracted.
4,2026-03-20T13:57:35.024820,10_breach_of_contract_nda.pdf,Loaded,Document loaded and text extracted.
...,...,...,...,...
325,2026-03-20 15:35:02,bw_doc_4 (8).pdf,Auto-Classification,Cease
326,2026-03-20 15:35:06,LOA5 (3).pdf,Manual Review,Cease
327,2026-03-20 15:35:11,LOA7 (1).pdf,Manual Review,Cease
328,2026-03-20 15:35:12,notice_1 (1).pdf,Auto-Classification,Cease
